In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_payments
# Source          : payments.csv
# Target          : procurement.silver.silver_payments
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned payments master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read employee data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower,upper,regexp_replace,to_date)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp

In [0]:
bronze_payments_df = spark.read.table(BRONZE_PAYMENTS)
display(bronze_payments_df)

In [0]:
# ============================================================
# Read Bronze Payments Table
# ============================================================

bronze_payments_df = read_delta(BRONZE_PAYMENTS)

preview(bronze_payments_df,"Bronze payments")

In [0]:
#============================================
# Create Sliver DataFrame
#===========================================
silver_payments_df = bronze_payments_df

In [0]:
# ============================================================
# Apply business transformations
# ============================================================

# Standardize payments_status as Matched
silver_payments_df = (silver_payments_df
    .withColumn("payment_id", trim(col("payment_id")))
    .withColumn("payment_date",to_date(trim(col("payment_date")), "yyyy-MM-dd"))
    .withColumn("invoice_id", trim(col("invoice_id")))
    .withColumn("amount_paid", col("amount_paid").cast(DoubleType()))
    .withColumn("currency", upper(trim(col("currency"))))
    .withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("payment_status", trim(col("payment_status")))
    .withColumn("days_delayed", col("days_delayed").cast(IntegerType()))
    )


In [0]:
# ============================================================
# Identify invalid payment records
# NULL & Blank payment_id
# NULL: payment_id,payment_date	invoice_id,amount_paid,currency,payment_method,payment_status,days_delayed
# ============================================================
invalid_payments = silver_payments_df.filter(
    (col("payment_id").isNull() | (trim(col("payment_id")) == ""))
    | (col("payment_date").isNull())
    | (col("invoice_id").isNull() | (trim(col("invoice_id")) == ""))
    | (col("amount_paid").isNull())  
    | (col("currency").isNull() | (trim(col("currency")) == ""))
    | (col("payment_method").isNull() | (trim(col("payment_method")) == ""))
    | (col("payment_status").isNull() | (trim(col("payment_status")) == ""))
    | (col("days_delayed").isNull())
)

print(f"Invalid Payment Records:{invalid_payments.count()}")
display(invalid_payments)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_payments = (invalid_payments.withColumn("audit_timestamp",current_timestamp())
    .withColumn("source_table",lit("payments"))
    .withColumn("pipeline_layer",lit("Silver"))
    .withColumn("issue_type",lit("Invalid Record")))

display(invalid_payments)

In [0]:
# ============================================================
# Write Invalid Payments to Audit Table
# ============================================================

if invalid_payments.count() > 0:
    write_delta(invalid_payments,AUDIT_INVALID_PAYMENTS,mode="overwrite")
    print("Invalid supplier records written.")
else:
    print("No invalid supplier records found.")

In [0]:
# ============================================================
# Remove Invalid Records
# ============================================================

silver_payments_df = silver_payments_df.filter(

    col("payment_id").isNotNull() & (trim(col("payment_id")) != "") &

    col("payment_date").isNotNull() &

    col("invoice_id").isNotNull() & (trim(col("invoice_id")) != "") &

    col("amount_paid").isNotNull() &

    col("currency").isNotNull() & (trim(col("currency")) != "") &

    col("payment_method").isNotNull() & (trim(col("payment_method")) != "") &

    col("payment_status").isNotNull() & (trim(col("payment_status")) != "") &

    col("days_delayed").isNotNull()
)

display(silver_payments_df)

In [0]:
# ============================================================
# Remove Duplicate Payments
# ============================================================

window_spec = Window.partitionBy("invoice_id").orderBy(col("payment_date").desc())

silver_payments_df = (silver_payments_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_payments_df,"Silver Payments")


In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_Payments_df = (silver_payments_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_payments_df,table_name=SILVER_PAYMENTS)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

bronze_count = bronze_payments_df.count()
invalid_count = invalid_payments.count()
silver_count = silver_payments_df.count()

duplicate_removed = bronze_count - invalid_count - silver_count

print("=" * 60)
print("Silver Payments Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records            : {bronze_count}")
print(f"Invalid Records Removed   : {invalid_count}")
print(f"Duplicate Records Removed : {duplicate_removed}")
print(f"Silver Records            : {silver_count}")